# M4.4: hybrid dense + sparse retrieval with RRF

This notebook compares frozen E5 dense, BM25 sparse, and rank-only Reciprocal Rank Fusion. It uses top-20 candidates from each retriever and primary `k=60`; it does not rerank or combine raw scores. If a CUDA failure occurred in this runtime, use **Runtime → Disconnect and delete runtime**, reopen the notebook, and Run All in a fresh session.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/ozgemelteminan/prompt-generator-rag'  # Replace this URL.
REPOSITORY_REF = 'main'  # Branch, tag, or commit to benchmark.
repository = Path('prompt-generator-rag')
if not repository.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL], check=True)
else:
    subprocess.run(['git', '-C', str(repository), 'fetch', '--all', '--tags', '--prune'], check=True)
subprocess.run(['git', '-C', str(repository), 'checkout', REPOSITORY_REF], check=True)
branch = subprocess.run(['git', '-C', str(repository), 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip()
if branch:
    subprocess.run(['git', '-C', str(repository), 'pull', '--ff-only', 'origin', branch], check=True)
os.chdir(repository)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'transformers==4.57.6', 'sentence-transformers==5.6.0'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'packages/prompt-engine'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'apps/api', '--no-deps'], check=True)

import torch
import transformers
import sentence_transformers
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('GPU:', GPU_NAME)
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('sentence-transformers:', sentence_transformers.__version__)
assert transformers.__version__ == '4.57.6'
RUNTIME_METADATA = {'torchVersion': torch.__version__, 'transformersVersion': transformers.__version__, 'sentenceTransformersVersion': sentence_transformers.__version__, 'cudaDevice': GPU_NAME}

repository_root = Path.cwd().resolve()
api_root = repository_root / 'apps' / 'api'
for import_root in (repository_root, api_root):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
stale_modules = [name for name in sys.modules if name == 'app' or name.startswith('app.') or name == 'evals' or name.startswith('evals.')]
if stale_modules:
    raise RuntimeError('Stale modules are loaded. Restart the runtime, then Run All.')

In [ ]:
import pandas as pd
from evals.src.dataset import load_dataset
from evals.src.embedding_eval import SentenceTransformerEmbeddingAdapter, embedding_model_registry, frozen_production_chunks
from evals.src.hybrid_eval import CANDIDATE_DEPTH, RRF_K, run_hybrid_benchmark, save_hybrid_results

ROOT = Path.cwd()
dataset = load_dataset(ROOT / 'evals/datasets/retrieval_eval_v1.json')
chunks = frozen_production_chunks(dataset)  # Generated once with 350/500/40 and reused everywhere.
e5_spec = embedding_model_registry()['multilingual_e5_large_instruct']
assert e5_spec.model_id == 'intfloat/multilingual-e5-large-instruct'
adapter = SentenceTransformerEmbeddingAdapter(e5_spec)
try:
    evaluation = run_hybrid_benchmark(dataset, chunks=chunks, adapter=adapter, rrf_k=RRF_K, candidate_depth=CANDIDATE_DEPTH)
finally:
    adapter.release()
save_hybrid_results(evaluation, dataset_version=dataset.version, output_dir=ROOT / 'evals/results/hybrid', runtime_metadata=RUNTIME_METADATA)
results = evaluation.results

In [ ]:
rows = []
for result in results:
    row = {'Retriever': result.retriever, **result.metrics}
    row['TR MRR'] = result.by_language.get('tr', {}).get('mrr', 0.0)
    row['EN MRR'] = result.by_language.get('en', {}).get('mrr', 0.0)
    rows.append(row)
comparison = pd.DataFrame(rows)
display(comparison)

hybrid = next(result for result in results if result.retriever_key == 'hybrid_rrf')
deltas = []
for baseline_key in ['dense_e5', 'sparse_bm25']:
    baseline = next(result for result in results if result.retriever_key == baseline_key)
    deltas.append({'Comparison': f'Hybrid - {baseline.retriever}', **{metric: hybrid.metrics[metric] - baseline.metrics[metric] for metric in hybrid.metrics}})
display(pd.DataFrame(deltas))

language_rows = []
for language in ['tr', 'en']:
    language_rows.append({'Language': language, **{result.retriever: result.by_language.get(language, {}).get('mrr', 0.0) for result in results}})
display(pd.DataFrame(language_rows))

category_rows = []
for category in ['factual', 'hard_paraphrase', 'terminology_mismatch', 'morphology_heavy', 'near_negative', 'same_topic_competitor', 'multi_section', 'cross_paragraph']:
    category_rows.append({'Category': category, **{result.retriever: result.by_category.get(category, {}).get('mrr', 0.0) for result in results}})
display(pd.DataFrame(category_rows))

for signal, query_ids in evaluation.diagnostics.items():
    print(f'{signal}: {len(query_ids)} queries', query_ids)